# Phần 1 — Xây dựng các mô hình phân loại

**Bài tập CO5085 — Deep Learning & Computer Vision | HCMUT 2025-2026**

Trong notebook này, ta xây dựng lần lượt 4 mô hình phân loại ảnh trên **CIFAR-100**:
1. **Softmax Regression** — mô hình tuyến tính đơn giản nhất
2. **MLP** — mạng fully connected nhiều lớp
3. **SimpleCNN** — mạng tích chập VGG-style
4. **SimpleViT** — Vision Transformer dùng PyTorch có sẵn

Dataset: CIFAR-100 (100 lớp, ảnh màu 32×32, 50,000 train / 10,000 test)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from src.data import get_cifar100_loaders, get_device
from src.models_part1 import SoftmaxRegression, MLP, SimpleCNN, SimpleViT
from src.utils import get_param_count

DEVICE = get_device()
print(f"Thiết bị: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

## 1. Tải và khám phá CIFAR-100

**Tại sao dùng CIFAR-100 thay vì CIFAR-10?**
- CIFAR-100: 100 lớp → bài toán khó hơn, phân biệt rõ hơn sức mạnh các mô hình
- CIFAR-10: chỉ 10 lớp → quá dễ, ngay cả Softmax Regression cũng đạt >80%

**Chuẩn hoá ảnh:** Dùng mean và std tính từ chính tập CIFAR-100,
*không phải* từ ImageNet (thường được dùng nhầm).

In [ ]:
train_loader, val_loader, test_loader, class_names = get_cifar100_loaders(batch_size=128)

print(f"\nSố lớp: {len(class_names)}")
print(f"Ví dụ một số lớp: {class_names[:10]}")

# Xem 1 batch ảnh mẫu
images, labels = next(iter(train_loader))
print(f"\nShape 1 batch: {images.shape}  (batch_size, channels, H, W)")
print(f"Label shape: {labels.shape}")
print(f"Giá trị pixel sau chuẩn hoá: min={images.min():.2f}, max={images.max():.2f}")

In [ ]:
# Visualize ảnh mẫu (cần denormalize để hiển thị đúng màu)
CIFAR100_MEAN = torch.tensor([0.5071, 0.4867, 0.4408])
CIFAR100_STD  = torch.tensor([0.2675, 0.2565, 0.2761])

def denormalize(img_tensor):
    """Chuyển từ normalized → [0,1] để hiển thị."""
    return (img_tensor * CIFAR100_STD[:, None, None] + CIFAR100_MEAN[:, None, None]).clamp(0, 1)

fig, axes = plt.subplots(3, 8, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(class_names[labels[i].item()], fontsize=7)
    ax.axis('off')
plt.suptitle('Mẫu ảnh từ CIFAR-100 (sau chuẩn hoá + denormalize để hiển thị)', fontsize=11)
plt.tight_layout()
plt.show()

## 2. Softmax Regression

**Ý tưởng:** Flatten ảnh thành vector 3072 chiều, ánh xạ tuyến tính đến 100 lớp.

```
[B, 3, 32, 32] → Flatten → [B, 3072] → Linear(3072, 100) → [B, 100]
```

**Lưu ý quan trọng:** Không cần thêm `softmax` trong `forward()` vì:
- `nn.CrossEntropyLoss` = `log_softmax` + `NLLLoss`
- Nó tự tính softmax bên trong khi tính loss
- Nếu thêm softmax rồi đưa vào CrossEntropyLoss → KẾT QUẢ SAI (double softmax)!

In [ ]:
model_softmax = SoftmaxRegression(num_classes=100)
print(model_softmax)
print(f"\nSố tham số: {get_param_count(model_softmax)}")

# Test forward pass
x_dummy = torch.randn(4, 3, 32, 32)
out = model_softmax(x_dummy)
print(f"\nInput shape:  {x_dummy.shape}")
print(f"Output shape: {out.shape}  (batch=4, num_classes=100)")
print(f"Output là logits (chưa qua softmax): min={out.min():.2f}, max={out.max():.2f}")

## 3. MLP (Multi-Layer Perceptron)

**Cải tiến so với Softmax:** Thêm các lớp ẩn với hàm kích hoạt ReLU
→ Học được các đặc trưng **phi tuyến**.

**Kiến trúc:**
```
Flatten → Linear(3072→512) → BatchNorm → ReLU → Dropout(0.3)
        → Linear(512→256)  → BatchNorm → ReLU → Dropout(0.3)
        → Linear(256→100)
```

**BatchNorm1d:** Chuẩn hoá output của mỗi lớp để training ổn định hơn.
**Dropout(0.3):** Tắt ngẫu nhiên 30% neurons → giảm overfitting.

**Nhược điểm của MLP:** `Flatten` làm mất thông tin về vị trí không gian (spatial).
Pixel (0,0) và pixel (31,31) được xử lý như các đặc trưng độc lập, không có quan hệ gần-xa.
→ CNN giải quyết vấn đề này với tích chập cục bộ.

In [ ]:
model_mlp = MLP(num_classes=100)
print(model_mlp)
print(f"\nSố tham số: {get_param_count(model_mlp)}")

out = model_mlp(x_dummy)
print(f"Output shape: {out.shape}")

## 4. SimpleCNN

**Ý tưởng tích chập (Convolution):**
- Mỗi bộ lọc (filter/kernel) kích thước 3×3 trượt qua ảnh
- Học được các đặc trưng cục bộ: cạnh, góc, texture, ...
- **Weight sharing**: cùng bộ lọc áp dụng tại mọi vị trí → ít params hơn
- **Translation invariance**: nhận diện mèo dù ở góc trái hay góc phải

**Kiến trúc (VGG-style):**
```
Conv(3→32)×2 + BN + ReLU + MaxPool(2)  → [B, 32, 16, 16]
Conv(32→64)×2 + BN + ReLU + MaxPool(2) → [B, 64, 8, 8]
Conv(64→128)×2 + BN + ReLU + MaxPool(2)→ [B, 128, 4, 4]
AdaptiveAvgPool(1) → Flatten → Linear(128→256) → ReLU → Linear(256→100)
```

**AdaptiveAvgPool(1):** Thay vì Flatten(128×4×4=2048) → Linear lớn,
pool về 128 features → ít tham số + ít overfitting.

In [ ]:
model_cnn = SimpleCNN(num_classes=100)
print(model_cnn)
print(f"\nSố tham số: {get_param_count(model_cnn)}")

# Trace shape qua từng bước
print("\n--- Shape trace qua CNN ---")
x = x_dummy
x_feat = model_cnn.features[0](x)  # Block 1
print(f"Sau Conv Block 1: {x_feat.shape}  (→ 32 feature maps, 16×16)")
x_feat = model_cnn.features[1](x_feat)  # Block 2
print(f"Sau Conv Block 2: {x_feat.shape}  (→ 64 feature maps, 8×8)")
x_feat = model_cnn.features[2](x_feat)  # Block 3
print(f"Sau Conv Block 3: {x_feat.shape}  (→ 128 feature maps, 4×4)")
out = model_cnn(x_dummy)
print(f"Output logits:    {out.shape}")

In [ ]:
# Visualize feature maps sau Conv Block 1
model_cnn.eval()
with torch.no_grad():
    feat_maps = model_cnn.features[0](images[:1])  # Chỉ lấy 1 ảnh

# feat_maps: [1, 32, 16, 16] → 32 feature maps
n_maps = 16
fig, axes = plt.subplots(2, n_maps // 2, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    if i < n_maps:
        fm = feat_maps[0, i].numpy()
        ax.imshow(fm, cmap='viridis')
        ax.set_title(f'Filter {i+1}', fontsize=7)
    ax.axis('off')
plt.suptitle('Feature maps sau Conv Block 1 (16 trong 32 filters)', fontsize=11)
plt.tight_layout()
plt.show()

## 5. SimpleViT — Vision Transformer

**Ý tưởng ViT:** Thay vì xử lý ảnh theo không gian (CNN),
chia ảnh thành các **patches** rồi xử lý như một chuỗi tokens (Transformer).

**Các bước:**
1. **Patch Embedding:** Chia 32×32 thành 64 patches 4×4
   - `Conv2d(3, 128, kernel_size=4, stride=4)` ≡ tích chập không chồng lấp = chia patch
2. **CLS Token:** Token đặc biệt học cách tổng hợp thông tin từ toàn ảnh
3. **Positional Encoding:** Cho model biết vị trí của mỗi patch (Transformer không có khái niệm thứ tự)
4. **Transformer Encoder:** Self-attention giữa tất cả 64 patches
5. **Classification:** Lấy CLS token → Linear → 100 lớp

**Tại sao patch_size=4 (không phải 16)?**
- ViT-B/16 dùng patch 16×16 vì được train trên ảnh 224×224 → 196 patches
- CIFAR-100 chỉ có 32×32: patch 16×16 → chỉ 4 patches — quá ít!
- Patch 4×4 → 64 patches — phù hợp hơn cho ảnh nhỏ

In [ ]:
model_vit = SimpleViT(num_classes=100)
print(model_vit)
print(f"\nSố tham số: {get_param_count(model_vit)}")

# Trace shape
print("\n--- Shape trace qua ViT ---")
x = x_dummy
patches = model_vit.patch_embed(x)
print(f"Sau patch embedding (Conv2d): {patches.shape}  ([B, d_model, 8, 8])")
patches_seq = patches.flatten(2).transpose(1, 2)
print(f"Sau reshape thành sequence:  {patches_seq.shape}  ([B, 64, 128])")
print(f"+ CLS token:                  [B, 65, 128]")
print(f"Sau Transformer:              [B, 65, 128]")
out = model_vit(x_dummy)
print(f"Output (từ CLS token):        {out.shape}")

In [ ]:
# Visualize 64 patches từ 1 ảnh
img = images[0]  # [3, 32, 32]
img_show = denormalize(img).permute(1, 2, 0).numpy()

# Chia thành 8×8 grid of patches 4×4
fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i in range(8):
    for j in range(8):
        patch = img_show[i*4:(i+1)*4, j*4:(j+1)*4, :]
        axes[i, j].imshow(patch)
        axes[i, j].axis('off')

plt.suptitle(f'64 patches 4×4 từ ảnh "{class_names[labels[0].item()]}"', fontsize=11)
plt.tight_layout()
plt.show()

## 6. Tổng kết — So sánh kiến trúc

| Mô hình | Kiến trúc | Số params | Inductive bias |
|---------|-----------|-----------|----------------|
| Softmax Regression | Flatten → Linear | ~307K | Không có |
| MLP | Flatten → FC × 2 → Linear | ~1.7M | Không có |
| SimpleCNN | Conv Blocks × 3 → FC | ~2.1M | Locality + Translation invariance |
| SimpleViT | Patch embed → Transformer | ~2.3M | Không có (học từ data) |

**Nhận xét:**
- CNN có **inductive bias** mạnh (biết rằng đặc trưng cục bộ quan trọng)
  → Train tốt với ít data hơn
- ViT **không có inductive bias** nhưng học được patterns phức tạp hơn
  → Cần nhiều data và epochs hơn để bắt kịp CNN

Tiếp theo: Notebook **Part 2** sẽ train và so sánh 4 mô hình này.

In [ ]:
# Bảng tóm tắt số params
models = {
    "SoftmaxRegression": model_softmax,
    "MLP": model_mlp,
    "SimpleCNN": model_cnn,
    "SimpleViT": model_vit,
}

print(f"{'Model':<25} | {'Params':>8}")
print("-" * 37)
for name, m in models.items():
    print(f"{name:<25} | {get_param_count(m):>8}")